# Device benchmarking & TensorRT deployment

Eager (FP32) anchors for Table 3, and the TensorRT FP16 two-engine deployment of
depth routing: BASE and SUPER are compiled as separate static engines (TRT cannot
skip layers within one engine), and the router selects which to run per frame. The
engines also emit the intermediate `tap` feature maps the router consumes.

In [ ]:
# --- setup: run from the repo root with the kernel's own python ---
import os, sys
while not os.path.isdir(f'{os.getcwd()}/method02_advantage_regress_tinyConv'):
    os.chdir('..')          # walk up to the anydepth-yolov12 repo root
PY = sys.executable
print('cwd =', os.getcwd()); print('python =', PY)

## Eager device anchors (Table 3)
`bench_device.py` (KITTI/BDD/Waymo, n=1000).

In [ ]:
!{PY} -m method02_advantage_regress_tinyConv.bench_device --n 1000 --warmup 40

## TensorRT: export → build → benchmark
Scripts live in `trt_bench/`. Order: PyTorch → ONNX → TRT engine (FP16).
Each detector path is exported with detection **and** raw tap maps (layers 4,6,8)
so the router can run on-device.

In [ ]:
# 1) export BASE/SUPER paths to ONNX (skip baked in) + raw tap maps
!{PY} trt_bench/export_onnx.py --weight finetuned_bdd100k/anydepth_best.pt \
    --imgsz 720 1280 --grid 2 --out_dir trt_bench/onnx/bdd

In [ ]:
# 2) build FP16 engines
!{PY} trt_bench/build_engine.py --onnx trt_bench/onnx/bdd/base.onnx --fp16
!{PY} trt_bench/build_engine.py --onnx trt_bench/onnx/bdd/super.onnx --fp16

In [ ]:
# 3a) pure-engine latency + switching overhead
!{PY} trt_bench/bench_trt.py --base trt_bench/onnx/bdd/base.fp16.engine \
    --super trt_bench/onnx/bdd/super.fp16.engine --iters 1000 --warmup 100

In [ ]:
# 3b) real-video threshold sweep: end-to-end realized SUPER% / latency / FPS / energy
!{PY} trt_bench/trt_video_eval.py --base trt_bench/onnx/bdd/base.fp16.engine \
    --super trt_bench/onnx/bdd/super.fp16.engine \
    --policy method02_advantage_regress_tinyConv/outputs/bdd100k/policy_scenario_s0.pt \
    --mot_root /media/data/bdd100k_mot/val --imgsz 720 1280 --limit 20